# Train DistilBERT 3-Class Issue Classifier — Local GPU

End-to-end training pipeline running **locally** on your GPU via the `ml/` uv environment.

**Structure:**
```
ml/
├── .venv/                       ← uv-managed Python env (torch+cu124)
├── data/                        ← raw_issues.jsonl, train/val/test splits
├── artifacts/
│   ├── classifier/best/         ← DistilBERT weights + model_card.json
│   ├── classical_ml/pipeline.joblib
│   └── eval_report.json
└── notebooks/
    ├── train_local.ipynb        ← this notebook
    └── train_classifier_colab.ipynb
```

**Secrets:** loaded from `project7/.env` (`GITHUB_TOKEN`, `GEMINI_API_KEY`)

## 0. Paths, Secrets, Imports

In [ ]:
import json
import os
import random
import time
import hashlib
import pickle
from collections import Counter, defaultdict
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import numpy as np

# ── paths ──────────────────────────────────────────────────────────────────────
ML_DIR       = Path.cwd().parent          # ml/
DATA_DIR     = ML_DIR / "data"
ARTIFACTS_DIR= ML_DIR / "artifacts"
BEST_DIR     = ARTIFACTS_DIR / "classifier" / "best"
CLASSICAL_DIR= ARTIFACTS_DIR / "classical_ml"

for d in (DATA_DIR, BEST_DIR, CLASSICAL_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"ML_DIR   : {ML_DIR}")
print(f"DATA_DIR : {DATA_DIR}")

# ── secrets ────────────────────────────────────────────────────────────────────
_env = ML_DIR.parent / ".env"
if _env.exists():
    with _env.open() as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                if k.strip() in ("GITHUB_TOKEN", "GEMINI_API_KEY"):
                    os.environ.setdefault(k.strip(), v.strip())

print(f"GITHUB_TOKEN  : {'✓ set' if os.environ.get('GITHUB_TOKEN') else '✗ missing'}")
print(f"GEMINI_API_KEY: {'✓ set' if os.environ.get('GEMINI_API_KEY') else '✗ missing (Gemini baseline will be skipped)'}")

## 1. Fetch MONAI Closed Issues

Checks `data/raw_issues.jsonl` first — skips the API call if the cache exists.

In [ ]:
import httpx

REPO       = "Project-MONAI/MONAI"
RAW_PATH   = DATA_DIR / "raw_issues.jsonl"

def fetch_closed_issues(force: bool = False) -> list[dict[str, Any]]:
    # ── Cache check — skip fetch if data already exists ─────────────────────
    if RAW_PATH.exists() and not force:
        with RAW_PATH.open() as fh:
            rows = [json.loads(line) for line in fh]
        print(f"✓ Cache hit: {len(rows)} issues from {RAW_PATH.name} (pass force=True to refetch)")
        return rows

    headers = {"Accept": "application/vnd.github+json"}
    if token := os.environ.get("GITHUB_TOKEN"):
        headers["Authorization"] = f"Bearer {token}"
        print("✓ Authenticated GitHub API (5000 req/hr)")
    else:
        print("⚠ Anonymous GitHub API (60 req/hr)")

    issues: list[dict[str, Any]] = []
    page = 1
    with httpx.Client(timeout=30.0, headers=headers) as client:
        while True:
            r = client.get(
                f"https://api.github.com/repos/{REPO}/issues",
                params={"state": "closed", "per_page": 100, "page": page},
            )
            r.raise_for_status()
            batch = r.json()
            if not batch:
                break
            issues.extend(i for i in batch if "pull_request" not in i)
            print(f"  page {page:>3}: +{len(batch):<3}  total={len(issues)}", flush=True)
            page += 1
            time.sleep(0.1)

    slim = [
        {
            "id": i["id"], "number": i["number"], "title": i["title"],
            "body": i.get("body"),
            "labels": [lb["name"] for lb in i.get("labels", [])],
            "created_at": i["created_at"], "closed_at": i["closed_at"],
        }
        for i in issues
    ]
    with RAW_PATH.open("w") as fh:
        for row in slim:
            fh.write(json.dumps(row) + "\n")
    print(f"✓ Saved {len(slim)} issues → {RAW_PATH.name}")
    return slim

raw_issues = fetch_closed_issues()

## 2. Label Mapping — 4 GitHub Labels → 3 Canonical Classes

`documentation + questions → support` (only 28 documentation examples; routing is identical).

In [ ]:
LABEL_MAP: dict[str, str] = {
    "bug": "bug",
    "Feature request": "feature",
    "feature request": "feature",
    "enhancement": "feature",
    "documentation": "support",
    "question": "support",
    "questions": "support",
}
CLASS_NAMES: tuple[str, ...] = ("bug", "feature", "support")
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
_TARGET_LABELS = frozenset(LABEL_MAP.keys())

def resolve_label(labels: list[str]) -> str | None:
    mapped = {LABEL_MAP[lbl] for lbl in labels if lbl in _TARGET_LABELS}
    return mapped.pop() if len(mapped) == 1 else None

def build_row(raw: dict[str, Any]) -> dict[str, Any] | None:
    label = resolve_label(raw["labels"])
    if label is None or raw.get("closed_at") is None:
        return None
    text = f"{raw['title']}\n\n{raw.get('body') or ''}".strip()
    return {"id": raw["id"], "number": raw["number"], "text": text,
            "label": label, "label_idx": CLASS_TO_IDX[label], "closed_at": raw["closed_at"]}

labeled = [r for r in (build_row(i) for i in raw_issues) if r is not None]
cc = Counter(r["label"] for r in labeled)
print(f"Labeled: {len(labeled)} issues")
for c in CLASS_NAMES:
    print(f"  {c:<10} {cc[c]:>5}")

## 1b. LLM Auto-Label — Gemini Labels Unlabeled Issues

Only ~50% of MONAI issues carry a recognised label tag.  
We send the remaining unlabeled issues to **Gemini 2.0 Flash** (5-shot, T=0) to nearly double the training corpus.  
Results are cached in `data/llm_labeled.jsonl` (committed) so the API is never called twice.  
The merged dataset is written to `data/processed_issues.jsonl` and replaces `labeled` for all downstream cells.

In [ ]:
import time as _time

PROCESSED_PATH = DATA_DIR / "processed_issues.jsonl"
LLM_CACHE_PATH  = DATA_DIR / "llm_labeled.jsonl"

# ── fast path: processed file already exists ───────────────────────────────
if PROCESSED_PATH.exists():
    with PROCESSED_PATH.open() as _f:
        _all = [json.loads(line) for line in _f]
    n_other = sum(1 for r in _all if r["label"] == "other")
    labeled = [r for r in _all if r["label"] != "other"]
    cc = Counter(r["label"] for r in labeled)
    print(f"✓ Cache hit: {len(_all)} total → {len(labeled)} training-eligible ({n_other} other filtered)")
    for c in CLASS_NAMES:
        print(f"  {c:<10} {cc[c]:>5}")
else:
    # ── identify issues with no/ambiguous label tags ───────────────────────
    unlabeled_raw = [
        i for i in raw_issues
        if resolve_label(i["labels"]) is None and i.get("closed_at") is not None
    ]
    print(f"  github-labeled : {len(labeled)}")
    print(f"  unlabeled raw  : {len(unlabeled_raw)}")

    # ── load LLM cache (may be partial from a previous interrupted run) ────
    llm_cache: dict[int, str] = {}
    if LLM_CACHE_PATH.exists():
        with LLM_CACHE_PATH.open() as _f:
            for line in _f:
                row = json.loads(line)
                llm_cache[row["id"]] = row["label"]
        print(f"  llm cache      : {len(llm_cache)} already labeled")

    to_label = [i for i in unlabeled_raw if i["id"] not in llm_cache]
    print(f"  still to label : {len(to_label)}")

    # ── call Gemini for remaining issues ───────────────────────────────────
    if to_label:
        _api_key = os.environ.get("GEMINI_API_KEY", "")
        if not _api_key:
            print("⚠  GEMINI_API_KEY not set — skipping LLM labeling")
        else:
            _GEMINI_MODEL = "gemini-2.0-flash"
            _K = 5
            _rng = random.Random(42)
            few_shot_ex: list[dict] = []
            for _cls in CLASS_NAMES:
                _pool = [r for r in labeled if r["label"] == _cls]
                _rng.shuffle(_pool)
                for _r in _pool[:_K]:
                    few_shot_ex.append({"text": _r["text"][:400], "label": _cls})
            random.Random(43).shuffle(few_shot_ex)

            _system = (
                "You are an issue classifier for open-source repositories.\n"
                "Classify the GitHub issue into exactly one of: bug, feature, support, other.\n\n"
                "  bug     -- A defect, crash, regression, or unexpected behaviour.\n"
                "  feature -- A request for new functionality or an enhancement.\n"
                "  support -- A question, docs gap, or request for help.\n"
                "  other   -- Does not clearly fit bug, feature, or support\n"
                "            (e.g. release notes, CI/infra-only, meta/admin issues).\n\n"
                "Reply with ONLY the single class name (lowercase, no punctuation, nothing else)."
            )

            def _make_contents(text: str) -> list[dict]:
                c: list[dict] = []
                for ex in few_shot_ex:
                    c.append({"role": "user",  "parts": [{"text": ex["text"]}]})
                    c.append({"role": "model", "parts": [{"text": ex["label"]}]})
                c.append({"role": "user", "parts": [{"text": text[:600]}]})
                return c

            _errors = 0
            with LLM_CACHE_PATH.open("a") as _cache_fh, httpx.Client(timeout=30.0) as _client:
                for _i, _issue in enumerate(to_label):
                    _text = f"{_issue['title']}\n\n{_issue.get('body') or ''}".strip()
                    try:
                        _r = _client.post(
                            f"https://generativelanguage.googleapis.com/v1beta/models/{_GEMINI_MODEL}:generateContent",
                            params={"key": _api_key},
                            json={
                                "system_instruction": {"parts": [{"text": _system}]},
                                "contents": _make_contents(_text),
                                "generationConfig": {"temperature": 0.0, "maxOutputTokens": 10},
                            },
                        )
                        _r.raise_for_status()
                        _raw_lbl = _r.json()["candidates"][0]["content"]["parts"][0]["text"].strip().lower()
                        _VALID_LABELS = set(CLASS_NAMES) | {"other"}
                        _label = _raw_lbl if _raw_lbl in _VALID_LABELS else "support"
                    except Exception as _e:
                        print(f"  error issue #{_issue['number']}: {_e}")
                        _errors += 1
                        _label = "support"

                    llm_cache[_issue["id"]] = _label
                    _cache_fh.write(json.dumps({"id": _issue["id"], "label": _label}) + "\n")
                    if (_i + 1) % 100 == 0:
                        print(f"  labeled {_i+1}/{len(to_label)} (errors={_errors})", flush=True)
                    _time.sleep(0.05)

            print(f"✓ Gemini labeling done  errors={_errors}")

    # ── build LLM-labeled rows ─────────────────────────────────────────────
    _id_to_raw = {i["id"]: i for i in raw_issues}
    llm_rows: list[dict] = []
    for _iid, _lbl in llm_cache.items():
        _raw = _id_to_raw.get(_iid)
        if _raw is None or _raw.get("closed_at") is None:
            continue
        _txt = f"{_raw['title']}\n\n{_raw.get('body') or ''}".strip()
        llm_rows.append({
            "id": _raw["id"], "number": _raw["number"],
            "text": _txt, "label": _lbl,
            "label_idx": CLASS_TO_IDX[_lbl], "closed_at": _raw["closed_at"],
            "label_source": "llm",
        })

    for _r in labeled:
        _r.setdefault("label_source", "github_label")

    labeled = labeled + llm_rows
    with PROCESSED_PATH.open("w") as _fh:
        for _r in labeled:
            _fh.write(json.dumps(_r) + "\n")

    cc = Counter(r["label"] for r in labeled)
    print(f"\n✓ Saved {len(labeled)} issues → processed_issues.jsonl")
    print(f"  github-label : {sum(1 for r in labeled if r.get('label_source') == 'github_label')}")
    print(f"  llm-labeled  : {sum(1 for r in labeled if r.get('label_source') == 'llm')}")
    for c in CLASS_NAMES:
        print(f"  {c:<10} {cc[c]:>5}")


## 3. Stratified Time-Aware Split (70 / 15 / 15)

Most-recent 15% by `closed_at` → **test** (prevents temporal leakage).

In [ ]:
RANDOM_SEED      = 42
TEST_FRACTION    = 0.15
VAL_OF_REMAINING = 0.15 / 0.85

rng = random.Random(RANDOM_SEED)
labeled_sorted = sorted(labeled, key=lambda r: r["closed_at"])

test_cut = int(len(labeled_sorted) * (1 - TEST_FRACTION))
remaining, test = labeled_sorted[:test_cut], labeled_sorted[test_cut:]

assert max(r["closed_at"] for r in remaining) <= min(r["closed_at"] for r in test), \
    "temporal leakage detected"

by_class: dict[str, list] = defaultdict(list)
for r in remaining:
    by_class[r["label"]].append(r)

train: list[dict] = []
val: list[dict] = []
for cls, rows in by_class.items():
    rng.shuffle(rows)
    n_val = max(1, int(len(rows) * VAL_OF_REMAINING))
    val.extend(rows[:n_val])
    train.extend(rows[n_val:])

rng.shuffle(train); rng.shuffle(val)

print("Split:")
for name, split in (("train", train), ("val", val), ("test", test)):
    cc2 = Counter(r["label"] for r in split)
    pct = len(split) / len(labeled) * 100
    print(f"  {name:<5} n={len(split):>5} ({pct:>4.1f}%)  " +
          "  ".join(f"{c}={cc2[c]}" for c in CLASS_NAMES))
    with (DATA_DIR / f"{name}.jsonl").open("w") as fh:
        for row in split:
            fh.write(json.dumps(row) + "\n")
print("✓ Splits saved")

## 4. Tokenize

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf(split: list[dict]) -> Dataset:
    ds = Dataset.from_dict({
        "text": [r["text"] for r in split],
        "label": [r["label_idx"] for r in split],
    })
    return ds.map(
        lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LENGTH, padding=False),
        batched=True,
    )

print("Tokenizing...")
train_ds = to_hf(train)
val_ds   = to_hf(val)
test_ds  = to_hf(test)
print(f"✓ train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

## 5. Train DistilBERT (3-class, EarlyStopping patience=2)

In [ ]:
import torch
import evaluate as ev
from transformers import (
    AutoModelForSequenceClassification, DataCollatorWithPadding,
    EarlyStoppingCallback, Trainer, TrainingArguments,
)

print(f"CUDA: {torch.cuda.is_available()}", end="")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"  GPU: {p.name}  VRAM: {p.total_memory/1e9:.1f} GB")
else:
    print(" — training on CPU (slow)")

CLASSIFIER_DIR = ARTIFACTS_DIR / "classifier"
BEST_DIR       = CLASSIFIER_DIR / "best"
BEST_DIR.mkdir(parents=True, exist_ok=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(CLASS_NAMES),
    id2label={i: c for i, c in enumerate(CLASS_NAMES)},
    label2id=CLASS_TO_IDX,
)

f1_metric = ev.load("f1")

def compute_metrics(ep):
    preds = np.argmax(ep.predictions, axis=-1)
    return f1_metric.compute(predictions=preds, references=ep.label_ids, average="macro")

training_args = TrainingArguments(
    output_dir=str(CLASSIFIER_DIR),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

t0 = time.time()
trainer.train()
print(f"\n✓ Training done in {(time.time()-t0)/60:.1f} min")

## 6. Save Best Checkpoint + Write `model_card.json`

In [ ]:
trainer.save_model(str(BEST_DIR))
tokenizer.save_pretrained(str(BEST_DIR))

def sha256_weights(directory: Path) -> str:
    h = hashlib.sha256()
    for pat in ("*.safetensors", "*.bin"):
        for p in sorted(directory.glob(pat)):
            with p.open("rb") as f:
                for chunk in iter(lambda: f.read(8192), b""):
                    h.update(chunk)
    return f"sha256:{h.hexdigest()}"

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return f"sha256:{h.hexdigest()}"

val_results = trainer.evaluate()
val_f1 = float(val_results.get("eval_f1", 0.0))

model_card = {
    "architecture": MODEL_NAME, "num_labels": len(CLASS_NAMES),
    "classes": list(CLASS_NAMES), "class_to_idx": CLASS_TO_IDX,
    "hyperparameters": {
        "learning_rate": training_args.learning_rate,
        "per_device_train_batch_size": training_args.per_device_train_batch_size,
        "num_train_epochs": training_args.num_train_epochs,
        "weight_decay": training_args.weight_decay,
        "warmup_ratio": training_args.warmup_ratio,
        "max_length": MAX_LENGTH,
    },
    "freeze_policy": "all layers unfrozen — full fine-tune of DistilBERT",
    "training_data_sha256": sha256_file(DATA_DIR / "train.jsonl"),
    "training_data_size": {"train": len(train), "val": len(val), "test": len(test)},
    "metrics": {"val_f1_macro": val_f1, "raw_eval": val_results},
    "model_sha256": sha256_weights(BEST_DIR),
    "trained_at": datetime.now(UTC).isoformat(),
    "version": "1.0.0",
    "dataset_repo": REPO,
}
(BEST_DIR / "model_card.json").write_text(json.dumps(model_card, indent=2))
print(f"✓ Saved to {BEST_DIR}")
print(f"  val macro-F1 : {val_f1:.4f}")
print(f"  model_sha256 : {model_card['model_sha256'][:30]}...")

## 6a. Evaluate DistilBERT on Test Split

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

test_labels = [r["label_idx"] for r in test]

t0 = time.perf_counter()
pred_out = trainer.predict(test_ds)
dl_latency_ms = (time.perf_counter() - t0) / len(test) * 1000

dl_preds     = np.argmax(pred_out.predictions, axis=-1).tolist()
dl_acc       = accuracy_score(test_labels, dl_preds)
dl_macro_f1  = f1_score(test_labels, dl_preds, average="macro")
dl_per_class = f1_score(test_labels, dl_preds, average=None).tolist()

print(f"DistilBERT — test results")
print(f"  Accuracy : {dl_acc:.4f}")
print(f"  Macro-F1 : {dl_macro_f1:.4f}")
print(f"  Per-class: {dict(zip(CLASS_NAMES, [f'{f:.4f}' for f in dl_per_class]))}")
print(f"  Latency  : {dl_latency_ms:.2f} ms/sample\n")
print(classification_report(test_labels, dl_preds, target_names=CLASS_NAMES))
print("Confusion matrix (row=true, col=pred):")
print(confusion_matrix(test_labels, dl_preds))

## 6b. Classical Baseline — TF-IDF + Logistic Regression

Saved as `.joblib` (more efficient than pickle for scikit-learn objects).

In [ ]:
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

train_texts  = [r["text"] for r in train]
train_labels = [r["label_idx"] for r in train]
test_texts   = [r["text"] for r in test]

ml_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=50_000,
                              sublinear_tf=True, strip_accents="unicode")),
    ("lr",    LogisticRegression(C=1.0, max_iter=1_000,
                                 random_state=RANDOM_SEED, class_weight="balanced")),
])
ml_pipeline.fit(train_texts, train_labels)

t0 = time.perf_counter()
ml_preds     = ml_pipeline.predict(test_texts).tolist()
ml_latency_ms= (time.perf_counter() - t0) / len(test) * 1000

ml_acc       = accuracy_score(test_labels, ml_preds)
ml_macro_f1  = f1_score(test_labels, ml_preds, average="macro")
ml_per_class = f1_score(test_labels, ml_preds, average=None).tolist()

# Save with joblib (better compression + faster than pickle for sklearn)
ml_path = CLASSICAL_DIR / "pipeline.joblib"
joblib.dump(ml_pipeline, ml_path, compress=3)

print(f"TF-IDF + LogReg — test results")
print(f"  Accuracy : {ml_acc:.4f}")
print(f"  Macro-F1 : {ml_macro_f1:.4f}")
print(f"  Per-class: {dict(zip(CLASS_NAMES, [f'{f:.4f}' for f in ml_per_class]))}")
print(f"  Latency  : {ml_latency_ms:.4f} ms/sample")
print(f"  Saved    : {ml_path}\n")
print(classification_report(test_labels, ml_preds, target_names=CLASS_NAMES))

## 6c. LLM Baseline — Gemini Few-Shot

Skipped automatically if `GEMINI_API_KEY` is not set.

In [ ]:
import random as _random

GEMINI_MODEL        = "gemini-2.0-flash"
GEMINI_EVAL_SAMPLE  = 100
K_SHOT              = 5
COST_PER_IN_TOK     = 0.10 / 1_000_000
COST_PER_OUT_TOK    = 0.40 / 1_000_000

llm_results: list[dict] = []
llm_acc = llm_macro_f1 = llm_avg_latency = llm_cost_per_1k = 0.0
llm_per_class: list[float] = [0.0] * len(CLASS_NAMES)

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

if not GEMINI_API_KEY:
    print("⚠ GEMINI_API_KEY not set — LLM baseline skipped")
else:
    rng2 = _random.Random(RANDOM_SEED)
    few_shot: list[dict] = []
    for cls in CLASS_NAMES:
        rows = [r for r in train if r["label"] == cls]
        rng2.shuffle(rows)
        for row in rows[:K_SHOT]:
            few_shot.append({"text": row["text"][:400].strip(), "label": cls})
    _random.Random(RANDOM_SEED + 1).shuffle(few_shot)

    SYSTEM = (
        "You are an issue classifier for open-source repositories.\n"
        "Classify the GitHub issue into exactly one of: bug, feature, support.\n\n"
        "  bug     -- A defect, crash, regression, or unexpected behaviour.\n"
        "  feature -- A request for new functionality or an enhancement.\n"
        "  support -- A question, docs gap, or request for help.\n\n"
        "Reply with ONLY the single class name (lowercase, no punctuation, nothing else)."
    )

    def _contents(text: str) -> list[dict]:
        c = []
        for ex in few_shot:
            c.append({"role": "user",  "parts": [{"text": ex["text"]}]})
            c.append({"role": "model", "parts": [{"text": ex["label"]}]})
        c.append({"role": "user", "parts": [{"text": text[:600]}]})
        return c

    def classify_gemini(text: str, client: httpx.Client) -> tuple[str, float, int, int]:
        body = {
            "system_instruction": {"parts": [{"text": SYSTEM}]},
            "contents": _contents(text),
            "generationConfig": {"temperature": 0.0, "maxOutputTokens": 10},
        }
        t0 = time.perf_counter()
        r = client.post(
            f"https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent",
            params={"key": GEMINI_API_KEY}, json=body, timeout=30.0,
        )
        lat = (time.perf_counter() - t0) * 1000
        r.raise_for_status()
        data = r.json()
        raw = data["candidates"][0]["content"]["parts"][0]["text"].strip().lower()
        label = raw if raw in CLASS_NAMES else "support"
        usage = data.get("usageMetadata", {})
        return label, lat, usage.get("promptTokenCount", 0), usage.get("candidatesTokenCount", 0)

    sample = test[:GEMINI_EVAL_SAMPLE]
    print(f"Evaluating {GEMINI_MODEL} on {len(sample)} examples ({K_SHOT}-shot per class)...")
    with httpx.Client() as client:
        for i, row in enumerate(sample):
            pred, lat, in_tok, out_tok = classify_gemini(row["text"], client)
            llm_results.append({"true": row["label"], "pred": pred,
                                 "latency_ms": lat, "in_tok": in_tok, "out_tok": out_tok})
            if (i + 1) % 10 == 0:
                print(f"  {i+1}/{len(sample)}", flush=True)
            time.sleep(0.1)

    true_idx = [CLASS_TO_IDX[r["true"]] for r in llm_results]
    pred_idx = [CLASS_TO_IDX.get(r["pred"], 2) for r in llm_results]
    llm_acc       = accuracy_score(true_idx, pred_idx)
    llm_macro_f1  = f1_score(true_idx, pred_idx, average="macro")
    llm_per_class = f1_score(true_idx, pred_idx, average=None).tolist()
    llm_avg_latency = sum(r["latency_ms"] for r in llm_results) / len(llm_results)
    avg_in  = sum(r["in_tok"] for r in llm_results) / len(llm_results)
    avg_out = sum(r["out_tok"] for r in llm_results) / len(llm_results)
    llm_cost_per_1k = (avg_in * COST_PER_IN_TOK + avg_out * COST_PER_OUT_TOK) * 1_000

    print(f"\n{GEMINI_MODEL} — test results (n={len(sample)})")
    print(f"  Accuracy : {llm_acc:.4f}")
    print(f"  Macro-F1 : {llm_macro_f1:.4f}")
    print(f"  Per-class: {dict(zip(CLASS_NAMES, [f'{f:.4f}' for f in llm_per_class]))}")
    print(f"  Latency  : {llm_avg_latency:.1f} ms/call")
    print(f"  Cost/1K  : ${llm_cost_per_1k:.4f}\n")
    print(classification_report(true_idx, pred_idx, target_names=CLASS_NAMES))

## 6d. Three-Way Comparison → `eval_report.json`

In [ ]:
THRESHOLDS = {"macro_f1": 0.78, "per_class_f1_min": 0.70}

candidates: dict[str, dict] = {
    "distilbert": {
        "accuracy": float(dl_acc), "macro_f1": float(dl_macro_f1),
        "per_class_f1": {c: float(f) for c, f in zip(CLASS_NAMES, dl_per_class)},
        "avg_latency_ms": float(dl_latency_ms),
        "cost_per_1k_predictions": 0.0, "eval_sample_size": len(test),
    },
    "tfidf_logreg": {
        "accuracy": float(ml_acc), "macro_f1": float(ml_macro_f1),
        "per_class_f1": {c: float(f) for c, f in zip(CLASS_NAMES, ml_per_class)},
        "avg_latency_ms": float(ml_latency_ms),
        "cost_per_1k_predictions": 0.0, "eval_sample_size": len(test),
    },
}
if llm_results:
    candidates[f"gemini_{GEMINI_MODEL.replace('-','_')}"] = {
        "accuracy": float(llm_acc), "macro_f1": float(llm_macro_f1),
        "per_class_f1": {c: float(f) for c, f in zip(CLASS_NAMES, llm_per_class)},
        "avg_latency_ms": float(llm_avg_latency),
        "cost_per_1k_predictions": float(llm_cost_per_1k),
        "eval_sample_size": len(llm_results),
    }

winner = max(candidates, key=lambda k: candidates[k]["macro_f1"])
wm = candidates[winner]
threshold_pass = (
    wm["macro_f1"] >= THRESHOLDS["macro_f1"]
    and all(v >= THRESHOLDS["per_class_f1_min"] for v in wm["per_class_f1"].values())
)

print("=== THREE-WAY COMPARISON ===\n")
print(f"{'Model':<35} {'Accuracy':>10} {'Macro-F1':>10} {'Latency ms':>12} {'$/1K':>10}")
print("-" * 80)
for name, m in sorted(candidates.items(), key=lambda kv: kv[1]["macro_f1"], reverse=True):
    marker = " ← WINNER" if name == winner else ""
    print(f"{name:<35} {m['accuracy']:>10.4f} {m['macro_f1']:>10.4f} "
          f"{m['avg_latency_ms']:>12.2f} ${m['cost_per_1k_predictions']:>9.4f}{marker}")

print(f"\nThreshold pass: {threshold_pass}")
if not threshold_pass:
    print("⚠  WARNING: winner below thresholds — inspect per-class F1")

eval_report = {
    "generated_at": datetime.now(UTC).isoformat(),
    "dataset_repo": REPO,
    "test_split_size": len(test),
    "llm_eval_sample_size": len(llm_results),
    "classes": list(CLASS_NAMES),
    "models": candidates,
    "winner": {
        "model": winner,
        "rationale": (
            f"Highest macro-F1 ({wm['macro_f1']:.4f}) on the time-aware test split. "
            "DistilBERT is self-hosted (zero per-call cost) and latency is acceptable "
            "for an interactive endpoint behind a model-server container."
        ),
        "metrics": wm,
    },
    "deployment_decision": winner,
    "thresholds": THRESHOLDS,
    "threshold_pass": threshold_pass,
}

out = ARTIFACTS_DIR / "eval_report.json"
out.write_text(json.dumps(eval_report, indent=2))
print(f"\n✓ eval_report.json → {out}")

## 7. Round-Trip Verify — Simulate `load_classifier()`

In [ ]:
EXPECTED_CLASSES = ("bug", "feature", "support")

card = json.loads((BEST_DIR / "model_card.json").read_text())

assert "classes" in card
assert tuple(card["classes"]) == EXPECTED_CLASSES, \
    f"class drift: {card['classes']} vs {list(EXPECTED_CLASSES)}"
assert card.get("model_sha256", "").startswith("sha256:"), "missing model_sha256"
assert sha256_weights(BEST_DIR) == card["model_sha256"], "sha256 mismatch"
assert (BEST_DIR / "tokenizer.json").exists() or (BEST_DIR / "tokenizer_config.json").exists(), \
    "tokenizer files missing"

print("✓ Artifact passes all load_classifier() checks")
print(f"  classes      : {card['classes']}")
print(f"  val_f1_macro : {card['metrics']['val_f1_macro']:.4f}")
print(f"  model_sha256 : {card['model_sha256'][:28]}...")